In [9]:

from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from scipy import sparse

from sklearn.linear_model import LogisticRegression

DATA_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/data/processed"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/data/feature_selection"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Processed data directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())
print("Output directory:", OUTPUT_DIR)


Processed data directory: /content/drive/MyDrive/Colab Notebooks/data/processed
Directory exists: True
Output directory: /content/drive/MyDrive/Colab Notebooks/data/feature_selection


In [11]:

feature_groups = ["Radiom", "RadioClin", "RadioClinLab"]

targets = [
    "ICU",
    "mechanical_ventilation",
    "death"
]

def load_matrix(name):
    npy_path = DATA_DIR / f"{name}.npy"
    npz_path = DATA_DIR / f"{name}.npz"

    if npy_path.exists():
        return np.load(npy_path)

    if npz_path.exists():
        return sparse.load_npz(npz_path)

    raise FileNotFoundError(
        f"Could not find {name}.npy or {name}.npz "
        f"in {DATA_DIR}"
    )

X_train_groups = {
    group: load_matrix(f"{group}_X_train")
    for group in feature_groups
}

X_test_groups = {
    group: load_matrix(f"{group}_X_test")
    for group in feature_groups
}

Y_train = pd.read_csv(DATA_DIR / "Y_train.csv")
Y_test = pd.read_csv(DATA_DIR / "Y_test.csv")

print("All matrices and labels loaded successfully.")


All matrices and labels loaded successfully.


In [12]:

preprocessors = {}

for group in feature_groups:
    preprocessor_path = (
        DATA_DIR / f"{group}_preprocessor.joblib"
    )

    if not preprocessor_path.exists():
        raise FileNotFoundError(
            f"Preprocessor not found: {preprocessor_path}"
        )

    preprocessors[group] = joblib.load(preprocessor_path)

    X_train = X_train_groups[group]
    X_test = X_test_groups[group]

    feature_names = preprocessors[
        group
    ].get_feature_names_out()

    assert X_train.shape[0] == len(Y_train)
    assert X_test.shape[0] == len(Y_test)
    assert X_train.shape[1] == X_test.shape[1]
    assert X_train.shape[1] == len(feature_names)

    print(
        f"{group}: "
        f"train={X_train.shape}, "
        f"test={X_test.shape}, "
        f"features={len(feature_names)}"
    )

assert all(target in Y_train.columns for target in targets)
assert all(target in Y_test.columns for target in targets)

print("\nData verification passed.")


Radiom: train=(1680, 120), test=(720, 120), features=120
RadioClin: train=(1680, 147), test=(720, 147), features=147
RadioClinLab: train=(1680, 164), test=(720, 164), features=164

Data verification passed.


In [13]:

selection_results = {}
selection_summary = []

for group in feature_groups:

    X_train = X_train_groups[group]
    X_test = X_test_groups[group]

    feature_names = preprocessors[
        group
    ].get_feature_names_out()

    for target in targets:

        y_train = Y_train[target].astype(int)

        print(f"\n{'=' * 55}")
        print(f"Feature group: {group}")
        print(f"Target: {target}")
        print("=" * 55)

        # L1-regularized logistic regression
        lasso_model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=1.0,
            max_iter=5000,
            random_state=42
        )

        # Fit using training data only
        lasso_model.fit(X_train, y_train)

        coefficients = lasso_model.coef_[0]
        absolute_coefficients = np.abs(coefficients)

        # Select features with non-zero coefficients
        selected_indices = np.flatnonzero(
            absolute_coefficients > 1e-8
        )

        # Fallback if LASSO selects no features
        if len(selected_indices) == 0:
            n_keep = min(10, len(coefficients))

            selected_indices = np.argsort(
                absolute_coefficients
            )[-n_keep:]

            print(
                "Warning: No non-zero coefficients. "
                f"Keeping the top {n_keep} features by "
                "absolute coefficient magnitude."
            )

        selected_features = feature_names[selected_indices]

        # Apply the same selected columns to train and test
        X_train_selected = X_train[:, selected_indices]
        X_test_selected = X_test[:, selected_indices]

        key = (group, target)

        selection_results[key] = {
            "X_train": X_train_selected,
            "X_test": X_test_selected,
            "selected_indices": selected_indices,
            "selected_features": selected_features,
            "model": lasso_model,
            "coefficients": coefficients
        }

        selection_summary.append({
            "Feature Group": group,
            "Target": target,
            "Original Features": X_train.shape[1],
            "Selected Features": len(selected_indices),
            "Features Removed": (
                X_train.shape[1] - len(selected_indices)
            )
        })

        print("Original features:", X_train.shape[1])
        print("Selected features:", len(selected_indices))
        print("Training shape:", X_train_selected.shape)
        print("Testing shape:", X_test_selected.shape)

print("\nAll nine feature-selection experiments completed.")



Feature group: Radiom
Target: ICU
Original features: 120
Selected features: 113
Training shape: (1680, 113)
Testing shape: (720, 113)

Feature group: Radiom
Target: mechanical_ventilation
Original features: 120
Selected features: 115
Training shape: (1680, 115)
Testing shape: (720, 115)

Feature group: Radiom
Target: death
Original features: 120
Selected features: 109
Training shape: (1680, 109)
Testing shape: (720, 109)

Feature group: RadioClin
Target: ICU
Original features: 147
Selected features: 131
Training shape: (1680, 131)
Testing shape: (720, 131)

Feature group: RadioClin
Target: mechanical_ventilation
Original features: 147
Selected features: 129
Training shape: (1680, 129)
Testing shape: (720, 129)

Feature group: RadioClin
Target: death
Original features: 147
Selected features: 128
Training shape: (1680, 128)
Testing shape: (720, 128)

Feature group: RadioClinLab
Target: ICU
Original features: 164
Selected features: 148
Training shape: (1680, 148)
Testing shape: (720, 148

In [14]:

def save_matrix(matrix, path):
    if sparse.issparse(matrix):
        sparse.save_npz(str(path) + ".npz", matrix)
    else:
        np.save(str(path) + ".npy", matrix)

for (group, target), result in selection_results.items():

    prefix = f"{group}_{target}"

    # Save selected training matrix
    save_matrix(
        result["X_train"],
        OUTPUT_DIR / f"{prefix}_X_train_selected"
    )

    # Save selected test matrix
    save_matrix(
        result["X_test"],
        OUTPUT_DIR / f"{prefix}_X_test_selected"
    )

    # Save selected column indices
    np.save(
        OUTPUT_DIR / f"{prefix}_selected_indices.npy",
        result["selected_indices"]
    )

    # Save selected feature names
    pd.Series(
        result["selected_features"],
        name="Feature"
    ).to_csv(
        OUTPUT_DIR / f"{prefix}_selected_features.csv",
        index=False
    )

    # Save fitted LASSO model
    joblib.dump(
        result["model"],
        OUTPUT_DIR / f"{prefix}_lasso_model.joblib"
    )

print("All selected feature sets and models saved.")
print("Output directory:", OUTPUT_DIR)


All selected feature sets and models saved.
Output directory: /content/drive/MyDrive/Colab Notebooks/data/feature_selection


In [15]:

summary_df = pd.DataFrame(selection_summary)

summary_df.to_csv(
    OUTPUT_DIR / "lasso_selection_summary.csv",
    index=False
)

display(summary_df)

print("\nSaved summary to:")
print(OUTPUT_DIR / "lasso_selection_summary.csv")


,Feature Group,Target,Original Features,Selected Features,Features Removed
0,Radiom,ICU,120,113,7
1,Radiom,mechanical_ventilation,120,115,5
2,Radiom,death,120,109,11
3,RadioClin,ICU,147,131,16
4,RadioClin,mechanical_ventilation,147,129,18
5,RadioClin,death,147,128,19
6,RadioClinLab,ICU,164,148,16
7,RadioClinLab,mechanical_ventilation,164,143,21
8,RadioClinLab,death,164,145,19



Saved summary to:
/content/drive/MyDrive/Colab Notebooks/data/feature_selection/lasso_selection_summary.csv


In [16]:

group = "RadioClinLab"
target = "ICU"

result = selection_results[(group, target)]

feature_names = preprocessors[
    group
].get_feature_names_out()

coefficients = result["coefficients"]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": np.abs(coefficients)
})

importance_df = importance_df.sort_values(
    "Absolute Coefficient",
    ascending=False
)

display(importance_df.head(10))


,Feature,Coefficient,Absolute Coefficient
100,numeric__wavelet_001,0.583487,0.583487
101,numeric__wavelet_002,0.473421,0.473421
50,numeric__glszm_001,0.403696,0.403696
138,categorical__sex_0,-0.318842,0.318842
156,categorical__copd_0,-0.253079,0.253079
123,numeric__lymphocyte,-0.252905,0.252905
143,categorical__cough_1,-0.244149,0.244149
64,numeric__glszm_015,-0.223963,0.223963
97,numeric__ngtdm_008,-0.222492,0.222492
144,categorical__dyspnea_0,-0.213271,0.213271


In [17]:

for (group, target), result in selection_results.items():

    assert (
        result["X_train"].shape[1]
        == result["X_test"].shape[1]
    )

    assert (
        result["X_train"].shape[0]
        == len(Y_train)
    )

    assert (
        result["X_test"].shape[0]
        == len(Y_test)
    )

    assert len(result["selected_indices"]) > 0

print("Verification passed for all nine experiments.")
print("Training and testing feature dimensions match.")
print("Test labels and test rows remain unchanged.")


Verification passed for all nine experiments.
Training and testing feature dimensions match.
Test labels and test rows remain unchanged.


In [18]:

saved_files = sorted(OUTPUT_DIR.iterdir())

print(f"Total saved files: {len(saved_files)}")

for file in saved_files:
    print(file.name)


Total saved files: 46
RadioClinLab_ICU_X_test_selected.npy
RadioClinLab_ICU_X_train_selected.npy
RadioClinLab_ICU_lasso_model.joblib
RadioClinLab_ICU_selected_features.csv
RadioClinLab_ICU_selected_indices.npy
RadioClinLab_death_X_test_selected.npy
RadioClinLab_death_X_train_selected.npy
RadioClinLab_death_lasso_model.joblib
RadioClinLab_death_selected_features.csv
RadioClinLab_death_selected_indices.npy
RadioClinLab_mechanical_ventilation_X_test_selected.npy
RadioClinLab_mechanical_ventilation_X_train_selected.npy
RadioClinLab_mechanical_ventilation_lasso_model.joblib
RadioClinLab_mechanical_ventilation_selected_features.csv
RadioClinLab_mechanical_ventilation_selected_indices.npy
RadioClin_ICU_X_test_selected.npy
RadioClin_ICU_X_train_selected.npy
RadioClin_ICU_lasso_model.joblib
RadioClin_ICU_selected_features.csv
RadioClin_ICU_selected_indices.npy
RadioClin_death_X_test_selected.npy
RadioClin_death_X_train_selected.npy
RadioClin_death_lasso_model.joblib
RadioClin_death_selected_fea